# MediNexus — Phase 1: Federated Learning for Medical-Shop Demand Forecasting

**Objective:** Train a shared medicine-demand forecasting model across multiple independent medical shops using Federated Learning (FedAvg), while keeping each shop's raw sales data local.

### Phase 1 outputs
1. 7-day and 30-day medicine demand forecasts
2. Centralized vs local vs federated model comparison
3. Expiry-risk and potential-wastage analysis
4. Stockout / overstock indicators
5. Recommended purchase quantity
6. Saved global model

> **Dataset note:** The uploaded Kaggle file is primarily a medicine/product catalog (medicine name, category, price, expiry and related metadata). It does not provide the historical daily sales series required for demand forecasting. Therefore this notebook uses the uploaded catalog as the medicine master and creates a reproducible synthetic two-year sales/inventory dataset for five medical shops. This is clearly labelled as synthetic experiment data, not real patient/customer data.


In [1]:
# Install dependencies (run this cell once in Google Colab if needed)
!pip -q install xgboost joblib


In [2]:
import os, warnings, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE = Path("..") if Path("../data/medicine_master.csv").exists() else Path(".")
DATA = BASE / "data"
MODELS = BASE / "models"
RESULTS = BASE / "results"
MODELS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

print("Data directory:", DATA.resolve())


Data directory: /mnt/data/medinexus_phase1_federated_learning/data


## 1. Load the medicine catalog and synthetic pharmacy datasets

In [4]:
medicine_master = pd.read_csv(DATA / "medicine_master.csv")
shops = pd.read_csv(DATA / "medical_shops.csv")
sales = pd.read_csv(DATA / "synthetic_shop_sales_2024_2025.csv", parse_dates=["date"])
inventory = pd.read_csv(DATA / "synthetic_shop_inventory_batches.csv", parse_dates=["expiry_date"])

print("Medicines:", len(medicine_master))
print("Medical shops:", len(shops))
print("Sales rows:", len(sales))
print("Inventory batches:", len(inventory))
display(medicine_master.head())
display(sales.head())


Medicines: 80
Medical shops: 5
Sales rows: 292400
Inventory batches: 800
  medicine_id medicine_name  ...  price expiry_date
0     MED0001     Metformin  ...   31.0  2025-04-10
1     MED0002    Cetirizine  ...   26.0  2025-05-15
2     MED0003    Tazarotene  ...  420.0  2025-02-02
3     MED0004     Steglatro  ...  500.0  2025-12-23
4     MED0005     Trazodone  ...  185.0  2025-06-15

[5 rows x 6 columns]
        date  shop_id medicine_id  ... week_of_year quarter  is_weekend
0 2024-01-01  SHOP_01     MED0001  ...            1       1           0
1 2024-01-02  SHOP_01     MED0001  ...            1       1           0
2 2024-01-03  SHOP_01     MED0001  ...            1       1           0
3 2024-01-04  SHOP_01     MED0001  ...            1       1           0
4 2024-01-05  SHOP_01     MED0001  ...            1       1           0

[5 rows x 11 columns]


## 2. Explore sales patterns

In [6]:
daily = sales.groupby("date", as_index=False)["sales_quantity"].sum()
plt.figure(figsize=(12,4))
plt.plot(daily["date"], daily["sales_quantity"])
plt.title("Synthetic Medical-Shop Sales Trend")
plt.xlabel("Date")
plt.ylabel("Units sold")
plt.grid(alpha=0.25)
plt.show()

top = sales.groupby("medicine_name")["sales_quantity"].sum().sort_values(ascending=False).head(15)
plt.figure(figsize=(10,5))
top.sort_values().plot(kind="barh")
plt.title("Top 15 Medicines by Sales")
plt.xlabel("Units sold")
plt.show()


## 3. Feature engineering

The forecasting model uses lagged demand, rolling demand, calendar/seasonal variables and shop/medicine identifiers. The target is daily `sales_quantity`.


In [8]:
sales = sales.sort_values(["shop_id","medicine_id","date"]).copy()

g = sales.groupby(["shop_id","medicine_id"], group_keys=False)

for lag in [1, 7, 14, 30]:
    sales[f"lag_{lag}"] = g["sales_quantity"].shift(lag)

# Compute rolling statistics per shop/medicine while preserving the original row order.
# Using groupby(...).transform avoids pandas MultiIndex differences across versions.
shifted = g["sales_quantity"].shift(1)
for window in [7, 30]:
    sales[f"rolling_mean_{window}"] = (
        shifted.groupby([sales["shop_id"], sales["medicine_id"]])
        .transform(lambda s: s.rolling(window, min_periods=1).mean())
    )
    sales[f"rolling_std_{window}"] = (
        shifted.groupby([sales["shop_id"], sales["medicine_id"]])
        .transform(lambda s: s.rolling(window, min_periods=2).std())
    )

# Encode categorical IDs consistently across all clients.
shop_codes = {x:i for i,x in enumerate(sorted(sales.shop_id.unique()))}
med_codes = {x:i for i,x in enumerate(sorted(sales.medicine_id.unique()))}
sales["shop_code"] = sales["shop_id"].map(shop_codes)
sales["medicine_code"] = sales["medicine_id"].map(med_codes)

feature_cols = [
    "shop_code","medicine_code","day_of_week","month","week_of_year","quarter","is_weekend",
    "lag_1","lag_7","lag_14","lag_30","rolling_mean_7","rolling_std_7",
    "rolling_mean_30","rolling_std_30"
]

model_df = sales.dropna(subset=feature_cols + ["sales_quantity"]).copy()
print("Model rows:", len(model_df))


Model rows: 280400


## 4. Time-based train/test split

No random shuffling is used for the final evaluation. The last 90 days are held out as future demand.


In [10]:
cutoff = model_df["date"].max() - pd.Timedelta(days=90)
train = model_df[model_df["date"] <= cutoff].copy()
test = model_df[model_df["date"] > cutoff].copy()

X_train, y_train = train[feature_cols], train["sales_quantity"]
X_test, y_test = test[feature_cols], test["sales_quantity"]

print("Train:", train.date.min().date(), "to", train.date.max().date())
print("Test :", test.date.min().date(), "to", test.date.max().date())


Train: 2024-01-31 to 2025-10-02
Test : 2025-10-03 to 2025-12-31


## 5. Baseline: centralized model

This model sees all shops' training rows. It is a benchmark only; the proposed system does not require raw sales data to be centralized.


In [12]:
central_model = XGBRegressor(
    n_estimators=180, max_depth=7, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    objective="reg:squarederror", random_state=RANDOM_STATE,
    n_jobs=2
)
central_model.fit(X_train, y_train)
central_pred = np.clip(central_model.predict(X_test), 0, None)

central_metrics = {
    "MAE": mean_absolute_error(y_test, central_pred),
    "RMSE": mean_squared_error(y_test, central_pred) ** 0.5,
    "R2": r2_score(y_test, central_pred),
}
central_metrics


## 6. Federated Learning setup

Each medical shop becomes a client. The server never loads the raw client training rows during aggregation.

For a tree model such as XGBoost, true parameter averaging is not naturally supported like it is for neural networks. Therefore, for this Phase 1 notebook we implement a **federated weighted ensemble**: each shop trains a local XGBoost model, and the global prediction is the sample-weighted average of local model predictions. This preserves the key FL data-flow experiment without pretending that XGBoost trees can be safely averaged as neural-network weights.

A later Phase 1B can replace this with a neural-network regressor and exact FedAvg parameter aggregation.


In [14]:
client_train = {
    shop_id: train[train.shop_id == shop_id].copy()
    for shop_id in sorted(train.shop_id.unique())
}
client_test = {
    shop_id: test[test.shop_id == shop_id].copy()
    for shop_id in sorted(test.shop_id.unique())
}

def train_local_model(df):
    model = XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.06,
        subsample=0.85, colsample_bytree=0.85,
        objective="reg:squarederror", random_state=RANDOM_STATE,
        n_jobs=2
    )
    model.fit(df[feature_cols], df["sales_quantity"])
    return model

local_models = {}
for shop_id, df in client_train.items():
    local_models[shop_id] = train_local_model(df)
    print(f"{shop_id}: trained on {len(df):,} rows")


SHOP_01: trained on 48,880 rows
SHOP_02: trained on 48,880 rows
SHOP_03: trained on 48,880 rows
SHOP_04: trained on 48,880 rows
SHOP_05: trained on 48,880 rows


## 7. Federated rounds

The ensemble is refreshed over several rounds. Each client trains locally using its own data; the server only combines model predictions. No client sales dataframe is sent to the aggregation function.


In [16]:
# Demonstration rounds. Local tree models are retrained from local data;
# the global model is represented by weighted prediction aggregation.
FED_ROUNDS = 3
global_history = []

client_sizes = {sid: len(df) for sid, df in client_train.items()}
total_client_rows = sum(client_sizes.values())
client_weights = {sid: n / total_client_rows for sid, n in client_sizes.items()}

global_pred_test = None
for rnd in range(1, FED_ROUNDS + 1):
    # Retrain each local model to emulate a federated round.
    for shop_id, df in client_train.items():
        local_models[shop_id] = train_local_model(df)

    # Aggregate only predictions on the evaluation rows.
    agg = np.zeros(len(test), dtype=float)
    for shop_id, model in local_models.items():
        agg += client_weights[shop_id] * np.clip(model.predict(test[feature_cols]), 0, None)

    global_pred_test = agg
    metrics = {
        "round": rnd,
        "MAE": mean_absolute_error(y_test, agg),
        "RMSE": mean_squared_error(y_test, agg) ** 0.5,
        "R2": r2_score(y_test, agg),
    }
    global_history.append(metrics)
    print(f"Round {rnd:02d} | MAE={metrics['MAE']:.3f} | RMSE={metrics['RMSE']:.3f} | R2={metrics['R2']:.3f}")

global_history = pd.DataFrame(global_history)
global_history.to_csv(RESULTS / "federated_round_metrics.csv", index=False)


Round 01 | MAE=2.865 | RMSE=3.799 | R2=0.831
Round 02 | MAE=2.865 | RMSE=3.799 | R2=0.831
Round 03 | MAE=2.865 | RMSE=3.799 | R2=0.831


## 8. Compare centralized, local and federated performance

In [18]:
local_rows = []
for shop_id, df in client_test.items():
    pred = np.clip(local_models[shop_id].predict(df[feature_cols]), 0, None)
    local_rows.append({
        "model": shop_id,
        "MAE": mean_absolute_error(df["sales_quantity"], pred),
        "RMSE": mean_squared_error(df["sales_quantity"], pred) ** 0.5,
        "R2": r2_score(df["sales_quantity"], pred),
    })

comparison = pd.DataFrame(local_rows)
fed_metrics = {
    "model": "Federated Global",
    "MAE": mean_absolute_error(y_test, global_pred_test),
    "RMSE": mean_squared_error(y_test, global_pred_test) ** 0.5,
    "R2": r2_score(y_test, global_pred_test),
}
cent_metrics = {"model":"Centralized", **central_metrics}
comparison = pd.concat([comparison, pd.DataFrame([cent_metrics, fed_metrics])], ignore_index=True)
display(comparison.round(3))
comparison.to_csv(RESULTS / "model_comparison.csv", index=False)


              model    MAE   RMSE     R2
0           SHOP_01  2.791  3.724  0.839
1           SHOP_02  2.723  3.618  0.823
2           SHOP_03  2.974  3.915  0.838
3           SHOP_04  2.947  3.867  0.816
4           SHOP_05  2.846  3.777  0.837
5       Centralized  2.848  3.757  0.835
6  Federated Global  2.865  3.799  0.831


## 9. Forecast future 30-day demand

For demonstration, we use the final observed window and recursively forecast each shop/medicine combination.


In [20]:
# Efficient recursive 30-day forecast using the federated local ensemble.
# Predictions are generated in batches per shop/day to keep execution practical.

history = model_df[["date","shop_id","medicine_id","sales_quantity"]].copy()
history = history.sort_values(["shop_id","medicine_id","date"])

# Keep only the recent demand history needed for recursive lag/rolling features.
hist_map = {}
for (sid, mid), grp in history.groupby(["shop_id", "medicine_id"]):
    hist_map[(sid, mid)] = list(grp["sales_quantity"].astype(float).tail(30))

forecast_start = model_df["date"].max() + pd.Timedelta(days=1)
future_rows = []

for shop_id in sorted(shop_codes):
    for day in range(30):
        d = forecast_start + pd.Timedelta(days=day)
        feature_rows = []
        keys = []
        for med_id in sorted(med_codes):
            vals = hist_map.get((shop_id, med_id), [0.0])
            def lag(k):
                return vals[-k] if len(vals) >= k else float(np.mean(vals[-min(len(vals), 7):]))
            r7 = np.array(vals[-7:]) if vals else np.array([0.0])
            r30 = np.array(vals[-30:]) if vals else np.array([0.0])
            feature_rows.append({
                "shop_code": shop_codes[shop_id],
                "medicine_code": med_codes[med_id],
                "day_of_week": d.dayofweek,
                "month": d.month,
                "week_of_year": int(d.isocalendar().week),
                "quarter": d.quarter,
                "is_weekend": int(d.dayofweek >= 5),
                "lag_1": lag(1), "lag_7": lag(7), "lag_14": lag(14), "lag_30": lag(30),
                "rolling_mean_7": float(r7.mean()),
                "rolling_std_7": float(r7.std() if len(r7)>1 else 0),
                "rolling_mean_30": float(r30.mean()),
                "rolling_std_30": float(r30.std() if len(r30)>1 else 0)
            })
            keys.append(med_id)

        Xn = pd.DataFrame(feature_rows)[feature_cols]
        pred = np.zeros(len(Xn), dtype=float)
        for sid, model in local_models.items():
            pred += client_weights[sid] * np.clip(model.predict(Xn), 0, None)

        for med_id, value in zip(keys, pred):
            value = float(value)
            future_rows.append((d, shop_id, med_id, value))
            hist_map.setdefault((shop_id, med_id), []).append(value)
            hist_map[(shop_id, med_id)] = hist_map[(shop_id, med_id)][-30:]

forecast = pd.DataFrame(future_rows, columns=["date","shop_id","medicine_id","predicted_sales"])
forecast["medicine_name"] = forecast["medicine_id"].map(
    medicine_master.set_index("medicine_id")["medicine_name"]
)
forecast_30 = forecast.groupby(["shop_id","medicine_id","medicine_name"], as_index=False)["predicted_sales"].sum()
forecast_30.to_csv(RESULTS / "30_day_demand_forecast.csv", index=False)
display(forecast_30.head(20))


    shop_id  ... predicted_sales
0   SHOP_01  ...      768.329250
1   SHOP_01  ...      631.301462
2   SHOP_01  ...      134.888002
3   SHOP_01  ...      648.446496
4   SHOP_01  ...      155.324213
5   SHOP_01  ...      744.905544
6   SHOP_01  ...      237.913054
7   SHOP_01  ...      426.052571
8   SHOP_01  ...      120.729330
9   SHOP_01  ...      143.152830
10  SHOP_01  ...      569.246946
11  SHOP_01  ...      636.055086
12  SHOP_01  ...      142.864887
13  SHOP_01  ...      657.702555
14  SHOP_01  ...      445.857119
15  SHOP_01  ...      223.139176
16  SHOP_01  ...      310.046359
17  SHOP_01  ...      929.531165
18  SHOP_01  ...      132.939823
19  SHOP_01  ...       96.003368

[20 rows x 4 columns]


## 10. Expiry-risk and wastage analysis

This is a deterministic decision layer on top of the forecast. It estimates whether current batch stock could exceed forecast consumption before expiry.

**Important:** This is an inventory-planning estimate. It should not be used as medical advice.


In [22]:
# Estimate 30-day demand per shop/medicine.
demand = forecast_30.rename(columns={"predicted_sales":"forecast_30d"})

# Aggregate current batch stock.
inv = inventory.groupby(["shop_id","medicine_id","medicine_name"], as_index=False).agg(
    current_stock=("opening_stock","sum"),
    nearest_expiry=("expiry_date","min")
)

analysis = inv.merge(demand, on=["shop_id","medicine_id","medicine_name"], how="left")
analysis["forecast_30d"] = analysis["forecast_30d"].fillna(0)
analysis["days_to_nearest_expiry"] = (
    pd.to_datetime(analysis["nearest_expiry"]) - forecast_start
).dt.days.clip(lower=0)

analysis["daily_forecast"] = analysis["forecast_30d"] / 30
analysis["forecast_before_expiry"] = np.minimum(
    analysis["forecast_30d"],
    analysis["daily_forecast"] * analysis["days_to_nearest_expiry"]
)
analysis["potential_excess"] = (
    analysis["current_stock"] - analysis["forecast_before_expiry"]
).clip(lower=0)

def expiry_risk(row):
    if row["potential_excess"] <= 0:
        return "LOW"
    if row["days_to_nearest_expiry"] <= 30 and row["potential_excess"] >= 0.25 * max(row["current_stock"],1):
        return "HIGH"
    if row["days_to_nearest_expiry"] <= 60 or row["potential_excess"] >= 0.40 * max(row["current_stock"],1):
        return "MEDIUM"
    return "LOW"

analysis["expiry_wastage_risk"] = analysis.apply(expiry_risk, axis=1)
analysis["stockout_risk"] = np.where(
    analysis["current_stock"] < analysis["forecast_30d"] * 0.5, "HIGH",
    np.where(analysis["current_stock"] < analysis["forecast_30d"], "MEDIUM", "LOW")
)

analysis.to_csv(RESULTS / "expiry_wastage_stockout_analysis.csv", index=False)
display(
    analysis.sort_values(["expiry_wastage_risk","potential_excess"], ascending=[True,False])
    .tail(20)
)


     shop_id medicine_id  ... expiry_wastage_risk  stockout_risk
384  SHOP_05     MED0065  ...                 LOW         MEDIUM
385  SHOP_05     MED0066  ...                 LOW         MEDIUM
386  SHOP_05     MED0067  ...                 LOW         MEDIUM
387  SHOP_05     MED0068  ...                 LOW           HIGH
388  SHOP_05     MED0069  ...                 LOW         MEDIUM
389  SHOP_05     MED0070  ...                 LOW           HIGH
390  SHOP_05     MED0071  ...                 LOW           HIGH
391  SHOP_05     MED0072  ...                 LOW         MEDIUM
392  SHOP_05     MED0073  ...                 LOW         MEDIUM
393  SHOP_05     MED0074  ...                 LOW           HIGH
394  SHOP_05     MED0075  ...                 LOW         MEDIUM
395  SHOP_05     MED0076  ...                 LOW         MEDIUM
396  SHOP_05     MED0077  ...                 LOW           HIGH
397  SHOP_05     MED0078  ...                 LOW         MEDIUM
398  SHOP_05     MED0079 

## 11. Purchase recommendation

A simple planning rule is used:

**recommended purchase = forecast demand + safety stock − current usable stock**

If the shop already has enough stock, the recommendation is zero. High expiry-risk stock suppresses unnecessary purchasing.


In [24]:
analysis["safety_stock"] = np.ceil(0.15 * analysis["forecast_30d"])
analysis["recommended_purchase"] = (
    analysis["forecast_30d"] + analysis["safety_stock"] - analysis["current_stock"]
).clip(lower=0).round().astype(int)

analysis.loc[
    analysis["expiry_wastage_risk"].eq("HIGH"),
    "recommended_purchase"
] = 0

analysis["recommendation"] = np.select(
    [
        analysis["expiry_wastage_risk"].eq("HIGH"),
        analysis["stockout_risk"].eq("HIGH"),
        analysis["recommended_purchase"].gt(0)
    ],
    [
        "DO NOT PURCHASE — prioritize existing/near-expiry stock and FEFO",
        "PURCHASE SOON — stock may be insufficient for forecast demand",
        "PURCHASE RECOMMENDED"
    ],
    default="NO ADDITIONAL PURCHASE NEEDED"
)

cols = [
    "shop_id","medicine_name","current_stock","forecast_30d",
    "days_to_nearest_expiry","potential_excess",
    "expiry_wastage_risk","stockout_risk",
    "recommended_purchase","recommendation"
]
display(analysis[cols].sort_values("potential_excess", ascending=False).head(25))
analysis[cols].to_csv(RESULTS / "purchase_recommendations.csv", index=False)


     shop_id  ...                                     recommendation
232  SHOP_03  ...  DO NOT PURCHASE — prioritize existing/near-exp...
217  SHOP_03  ...  DO NOT PURCHASE — prioritize existing/near-exp...
221  SHOP_03  ...  DO NOT PURCHASE — prioritize existing/near-exp...
347  SHOP_05  ...  DO NOT PURCHASE — prioritize existing/near-exp...
179  SHOP_03  ...  DO NOT PURCHASE — prioritize existing/near-exp...
246  SHOP_04  ...  DO NOT PURCHASE — prioritize existing/near-exp...
106  SHOP_02  ...                               PURCHASE RECOMMENDED
266  SHOP_04  ...                               PURCHASE RECOMMENDED
274  SHOP_04  ...                               PURCHASE RECOMMENDED
267  SHOP_04  ...                               PURCHASE RECOMMENDED
15   SHOP_01  ...                               PURCHASE RECOMMENDED
331  SHOP_05  ...                               PURCHASE RECOMMENDED
134  SHOP_02  ...                               PURCHASE RECOMMENDED
270  SHOP_04  ...                 

## 12. FEFO batch view

For retail pharmacy inventory, First-Expire-First-Out (FEFO) is useful for operational planning: batches with earlier expiry dates should be prioritized for sale where legally and operationally appropriate.


In [26]:
fefo = inventory.sort_values(["shop_id","medicine_id","expiry_date"]).copy()
fefo["days_to_expiry"] = (fefo["expiry_date"] - forecast_start).dt.days
display(fefo[["shop_id","medicine_name","batch_id","opening_stock","expiry_date","days_to_expiry"]].head(30))
fefo.to_csv(RESULTS / "fefo_batch_priority.csv", index=False)


     shop_id                 medicine_name  ... expiry_date  days_to_expiry
0    SHOP_01                     Metformin  ...  2027-05-07             491
1    SHOP_01                     Metformin  ...  2027-11-16             684
11   SHOP_01                    Cetirizine  ...  2027-02-14             409
10   SHOP_01                    Cetirizine  ...  2027-10-31             668
21   SHOP_01                    Tazarotene  ...  2027-04-20             474
20   SHOP_01                    Tazarotene  ...  2027-10-23             660
31   SHOP_01                     Steglatro  ...  2027-01-23             387
30   SHOP_01                     Steglatro  ...  2027-11-16             684
40   SHOP_01                     Trazodone  ...  2027-12-12             710
41   SHOP_01                     Trazodone  ...  2027-12-25             723
50   SHOP_01                    Alogliptin  ...  2026-02-20              50
51   SHOP_01                    Alogliptin  ...  2027-05-08             492
60   SHOP_01

## 13. Save the trained models and experiment metadata

In [28]:
# Save each local model plus the final federated ensemble metadata.
for shop_id, model in local_models.items():
    joblib.dump(model, MODELS / f"local_model_{shop_id}.joblib")

metadata = {
    "random_state": RANDOM_STATE,
    "federated_rounds": FED_ROUNDS,
    "shops": sorted(shop_codes),
    "medicine_count": len(med_codes),
    "feature_columns": feature_cols,
    "aggregation": "sample-weighted prediction ensemble (FedAvg-style demonstration for tree models)",
    "note": "Raw shop sales remain in local client partitions during the aggregation step."
}
joblib.dump(metadata, MODELS / "federated_model_metadata.joblib")

# Save the centralized benchmark too.
joblib.dump(central_model, MODELS / "centralized_xgboost_benchmark.joblib")

print("Saved models to:", MODELS.resolve())
print("Saved results to:", RESULTS.resolve())


Saved models to: /mnt/data/medinexus_phase1_federated_learning/models
Saved results to: /mnt/data/medinexus_phase1_federated_learning/results


## 14. Final summary

The notebook demonstrates:

- Multiple independent **medical shops**
- Private/local shop datasets
- Local demand-model training
- Federated aggregation without sending raw sales tables to the server
- Centralized benchmark comparison
- 30-day demand forecasting
- Expiry/wastage risk
- Stockout risk
- Purchase quantity recommendation
- FEFO batch prioritization
- Saved trained models

### Next phase
For a stricter research implementation, replace the tree-based federated ensemble with a neural-network time-series model (for example, an MLP/LSTM) and perform **exact FedAvg parameter aggregation** across clients. Then add secure aggregation and differential privacy only after the basic FL pipeline is validated.
